In [5]:
import mlflow

mlflow.set_tracking_uri(
    "http://ec2-54-234-203-17.compute-1.amazonaws.com:5000"
)

with mlflow.start_run():
    mlflow.log_param("param1", 10)
    mlflow.log_metric("metric1", 1.0)


🏃 View run inquisitive-crab-809 at: http://ec2-54-234-203-17.compute-1.amazonaws.com:5000/#/experiments/0/runs/68902a0d2ce54a0a9dd85c30cba6dcf0
🧪 View experiment at: http://ec2-54-234-203-17.compute-1.amazonaws.com:5000/#/experiments/0


In [6]:
import pandas as pd


In [7]:
df=pd.read_csv('/Users/siddharthaganguli/Desktop/proj3/youtube_Comment_analyzer/data/preproceesed/basic_preprocessed.csv')


In [8]:
df.head()


,clean_comment,category,word_count
0,family mormon have never tried explain them th...,1,39
1,buddhism has very much lot compatible with chr...,1,196
2,seriously don say thing first all they won get...,-1,86
3,what you have learned yours and only yours wha...,0,29
4,for your own benefit you may want read living ...,1,112


In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [10]:
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/siddharthaganguli/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/siddharthaganguli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment


In [12]:
df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)


In [13]:
import tempfile
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [14]:
vectorizer = CountVectorizer(max_features=10000)


In [15]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category'] 


In [16]:
mlflow.set_experiment("RF Baseline")


<Experiment: artifact_location='s3://yt-sentiment-mlflow-rik/1', creation_time=1781896310243, experiment_id='1', last_update_time=1781896310243, lifecycle_stage='active', name='RF Baseline', tags={}>

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [19]:
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

# Start MLflow run
with mlflow.start_run(run_name="RandomForest_Baseline_TrainTestSplit"):

    # Tags
    mlflow.set_tag("experiment_type", "baseline")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag(
        "description",
        "Baseline RandomForest model for sentiment analysis using Bag of Words (BoW) with a simple train-test split"
    )

    # Vectorizer Parameters
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("vectorizer_max_features", vectorizer.max_features)

    # Random Forest Parameters
    n_estimators = 200
    max_depth = 15

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", 42)

    # Model Training
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    # Classification Report
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric_name}",
                    float(metric_value)
                )

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues"
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Random Forest Confusion Matrix")

    plt.tight_layout()

    cm_path = "confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()

    # Log artifact
    mlflow.log_artifact(cm_path)

    # Log Model (new MLflow syntax)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="random_forest_model"
    )

print(f"Accuracy: {accuracy:.4f}")


2026/06/22 01:37:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run RandomForest_Baseline_TrainTestSplit at: http://ec2-54-234-203-17.compute-1.amazonaws.com:5000/#/experiments/1/runs/eb8da0e10f2d4e408adc50300808255f
🧪 View experiment at: http://ec2-54-234-203-17.compute-1.amazonaws.com:5000/#/experiments/1
Accuracy: 0.6483


In [ ]:
import mlflow 

print(mlflow.__version__)


3.14.0
